# Baseline Classification Models

This notebook trains baseline classification models using the selected RDKit molecular descriptors. Logistic Regression, Random Forest, and HistGradientBoosting classifiers are trained on the official training split and evaluated on the validation split using classification metrics and prediction probabilities. The held-out scaffold test set is not used at this stage.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_sample_weight


DATA_FILE = "preprocessed_model_data.csv"
DESCRIPTOR_FILE = "selected_descriptor_names.csv"
RANDOM_SEED = 42


data_df = pd.read_csv(DATA_FILE)

descriptor_columns = pd.read_csv(
    DESCRIPTOR_FILE
)["descriptor"].tolist()


print("Dataset shape:", data_df.shape)
print("Number of descriptors:", len(descriptor_columns))

In [ ]:
# Prepare official training and validation data

train_df = data_df[
    data_df["split"] == "train"
].copy()

validation_df = data_df[
    data_df["split"] == "validation"
].copy()


X_train = train_df[descriptor_columns]
y_train = train_df["dual_candidate"]

X_validation = validation_df[descriptor_columns]
y_validation = validation_df["dual_candidate"]


print("Training molecules:", len(train_df))
print("Validation molecules:", len(validation_df))

print("\nTraining dual percentage:")
print(round(y_train.mean() * 100, 2))

print("\nValidation dual percentage:")
print(round(y_validation.mean() * 100, 2))

print("\nThe test set has not been used.")

In [ ]:
# Create the classification models

logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_SEED
        )
    )
])


random_forest_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1
)


gradient_boosting_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    random_state=RANDOM_SEED
)


# Weights help Gradient Boosting handle the class imbalance
gradient_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)


models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}

print("Models created:")
print(list(models.keys()))

In [ ]:
# Train the models and save their predictions

trained_models = {}
validation_predictions = {}


for model_name, model in models.items():

    print("Training:", model_name)

    if model_name == "Gradient Boosting":
        model.fit(
            X_train,
            y_train,
            sample_weight=gradient_weights
        )
    else:
        model.fit(
            X_train,
            y_train
        )

    predicted_classes = model.predict(
        X_validation
    )

    predicted_probabilities = model.predict_proba(
        X_validation
    )[:, 1]

    trained_models[model_name] = model

    validation_predictions[model_name] = {
        "classes": predicted_classes,
        "probabilities": predicted_probabilities
    }

    # Save each trained model
    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
    )

    joblib.dump(
        model,
        f"{safe_name}_classification_model.joblib"
    )

    print(model_name, "completed.\n")


print("All classification models trained successfully.")

In [ ]:
#  Evaluate and compare the models

results = []
prediction_tables = []


for model_name in models.keys():

    predicted_classes = validation_predictions[
        model_name
    ]["classes"]

    predicted_probabilities = validation_predictions[
        model_name
    ]["probabilities"]

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predicted_classes
    ).ravel()

    specificity = tn / (tn + fp)

    results.append({
        "model": model_name,

        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            predicted_classes
        ),

        "mcc": matthews_corrcoef(
            y_validation,
            predicted_classes
        ),

        "roc_auc": roc_auc_score(
            y_validation,
            predicted_probabilities
        ),

        "average_precision": average_precision_score(
            y_validation,
            predicted_probabilities
        ),

        "precision": precision_score(
            y_validation,
            predicted_classes,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            predicted_classes,
            zero_division=0
        ),

        "specificity": specificity,

        "f1_score": f1_score(
            y_validation,
            predicted_classes,
            zero_division=0
        ),

        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })


    model_predictions = pd.DataFrame({
        "canonical_smiles": validation_df[
            "canonical_smiles"
        ].values,

        "true_label": y_validation.values,

        "model": model_name,

        "predicted_class": predicted_classes,

        "predicted_probability": predicted_probabilities
    })

    prediction_tables.append(model_predictions)


classification_results_df = pd.DataFrame(
    results
).sort_values(
    by="average_precision",
    ascending=False
)

validation_predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True
)


display(
    classification_results_df.round(3)
)


classification_results_df.to_csv(
    "classification_validation_results.csv",
    index=False
)

validation_predictions_df.to_csv(
    "classification_validation_predictions.csv",
    index=False
)


print("Saved: classification_validation_results.csv")
print("Saved: classification_validation_predictions.csv")